In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("diabetes_dataset_unclean.csv")


In [ ]:
df.shape

In [ ]:
df = df.drop_duplicates()


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
categorical_cols = df.select_dtypes(
    include=["object"]
).columns.tolist()

In [ ]:
numerical_cols = df.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

In [ ]:
extra_categorical = [
    'family_history_diabetes',
    'hypertension_history',
    'cardiovascular_history',
    'diabetes_stage'
]

In [ ]:
categorical_cols.extend(extra_categorical)


In [ ]:
categorical_cols = list(set(categorical_cols))


In [ ]:
numerical_cols = [
    col for col in numerical_cols
    if col not in extra_categorical
]

In [ ]:
print("Categorical Columns:")
print(categorical_cols)

print("\nNumerical Columns:")
print(numerical_cols)

In [ ]:
target_col = "diagnosed_diabetes"
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

In [19]:
num_imputer = KNNImputer(n_neighbors=5)

df[numerical_cols] = num_imputer.fit_transform(
    df[numerical_cols]
)

In [20]:
if len(categorical_cols) > 0:

    cat_imputer = SimpleImputer(
        strategy="most_frequent"
    )

    df[categorical_cols] = cat_imputer.fit_transform(
        df[categorical_cols]
    )

In [21]:
print("Categorical Columns:")
print(categorical_cols)

print("\nNumerical Columns:")
print(numerical_cols)

Categorical Columns:
['family_history_diabetes', 'smoking_status', 'ethnicity', 'diabetes_stage', 'employment_status', 'hypertension_history', 'income_level', 'gender', 'cardiovascular_history', 'education_level']

Numerical Columns:
['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c', 'diabetes_risk_score']


In [22]:
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [23]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[numerical_cols] = scaler.fit_transform(
    df[numerical_cols]
)


In [24]:

df.to_csv(
    "diabetes_dataset_cleaned.csv",
    index=False
)

## model training

In [25]:
import pandas as pd 
df=pd.read_csv("diabetes_dataset_cleaned.csv")

In [26]:
X = df.drop(["diagnosed_diabetes","diabetes_risk_score"],axis=1)
y = df["diagnosed_diabetes"]

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (80058, 47)
X_test : (20015, 47)


In [28]:
print(y_train.value_counts())

print(
    y_train.value_counts(normalize=True) * 100
)

diagnosed_diabetes
1    48033
0    32025
Name: count, dtype: int64
diagnosed_diabetes
1    59.997752
0    40.002248
Name: proportion, dtype: float64


In [29]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:,1]

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.9997501873594804
Precision: 1.0
Recall   : 0.9995836109260493
F1 Score : 0.9997917621090333
ROC-AUC  : 0.9999986583192072

Confusion Matrix
[[ 8007     0]
 [    5 12003]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8007
           1       1.00      1.00      1.00     12008

    accuracy                           1.00     20015
   macro avg       1.00      1.00      1.00     20015
weighted avg       1.00      1.00      1.00     20015



In [31]:
import pickle

In [36]:
with open("diabetes_model.pkl", "wb") as f:
    pickle.dump(rf, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

model_columns = list(X_train.columns)
with open("model_columns.pkl", "wb") as f:
    pickle.dump(model_columns, f)